In [4]:
import os
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [28]:
# Paths
TRAIN_PATH = "../data/train_dataset.csv"
OUT_DIR = "../models"
OUT_PARAMS_CSV = os.path.join(OUT_DIR, "svr_refined_best_params.csv")
OUT_PARAMS_JSON = os.path.join(OUT_DIR, "svr_refined_best_params.json")

os.makedirs(OUT_DIR, exist_ok=True)

# Config
DATE_COL = "date"
TARGET_COLS = ["y_h1", "y_h3", "y_h5", "y_h7"]

# Validation split: last 20% of training set
VAL_RATIO = 0.2

# stage 1 tuning grid
PARAM_GRID = [
    {
        "svr__kernel": ["linear"],
        "svr__C": [0.1, 1, 10],
        "svr__epsilon": [0.01, 0.1]
    },
    {
        "svr__kernel": ["rbf"],
        "svr__C": [1, 10, 50],
        "svr__epsilon": [0.01, 0.1],
        "svr__gamma": ["scale"]
    }
]


In [6]:
# Load data
df = pd.read_csv(TRAIN_PATH)
df[DATE_COL] = pd.to_datetime(df["date"], format="%d/%m/%y", errors="raise")
df = df.sort_values(DATE_COL).reset_index(drop=True)

# Feature columns = everything except date and targets
feature_cols = [c for c in df.columns if c not in [DATE_COL] + TARGET_COLS]

print("Train data shape:", df.shape)
print("Number of features:", len(feature_cols))
print("Date range:", df[DATE_COL].min(), "to", df[DATE_COL].max())

Train data shape: (1268, 181)
Number of features: 176
Date range: 2021-01-08 00:00:00 to 2024-06-28 00:00:00


In [7]:
# Train / validation split
cut = int(len(df) * (1 - VAL_RATIO))
train_sub = df.iloc[:cut].copy()
val_sub = df.iloc[cut:].copy()

print("\nSub-train rows:", len(train_sub))
print("Validation rows:", len(val_sub))
print("Validation starts from:", val_sub[DATE_COL].min())

X_train = train_sub[feature_cols]
X_val = val_sub[feature_cols]

results = []
best_params_all = {}



Sub-train rows: 1014
Validation rows: 254
Validation starts from: 2023-10-19 00:00:00


In [ ]:
# Stage 1 tuning: linear and rbf kernels with limited C and epsilon values

for target_col in TARGET_COLS:
    print(f"\n===== Tuning {target_col} =====")

    y_train = train_sub[target_col]
    y_val = val_sub[target_col]

    best_rmse = np.inf
    best_result = None

    for i, params in enumerate(ParameterGrid(PARAM_GRID), start=1):
        print(f"{target_col} | combo {i}")
        
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR())
        ])

        model.set_params(**params)
        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        mae = mean_absolute_error(y_val, val_pred)

        row = {
            "target": target_col,
            "rmse_val": rmse,
            "mae_val": mae,
            **params
        }
        results.append(row)

        if rmse < best_rmse:
            best_rmse = rmse
            best_result = row

    best_params_all[target_col] = {
        k: v for k, v in best_result.items()
        if k.startswith("svr__")
    }

    print("Best validation RMSE:", round(best_result["rmse_val"], 6))
    print("Best validation MAE :", round(best_result["mae_val"], 6))
    print("Best params         :", best_params_all[target_col])


Sub-train rows: 1014
Validation rows: 254
Validation starts from: 2023-10-19 00:00:00

===== Tuning y_h1 =====
y_h1 | combo 1
y_h1 | combo 2
y_h1 | combo 3
y_h1 | combo 4
y_h1 | combo 5
y_h1 | combo 6
y_h1 | combo 7
y_h1 | combo 8
y_h1 | combo 9
y_h1 | combo 10
y_h1 | combo 11
y_h1 | combo 12
Best validation RMSE: 0.78969
Best validation MAE : 0.611787
Best params         : {'svr__C': 0.1, 'svr__epsilon': 0.1, 'svr__kernel': 'linear'}

===== Tuning y_h3 =====
y_h3 | combo 1
y_h3 | combo 2
y_h3 | combo 3
y_h3 | combo 4
y_h3 | combo 5
y_h3 | combo 6
y_h3 | combo 7
y_h3 | combo 8
y_h3 | combo 9
y_h3 | combo 10
y_h3 | combo 11
y_h3 | combo 12
Best validation RMSE: 0.877512
Best validation MAE : 0.665275
Best params         : {'svr__C': 1, 'svr__epsilon': 0.1, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}

===== Tuning y_h5 =====
y_h5 | combo 1
y_h5 | combo 2
y_h5 | combo 3
y_h5 | combo 4
y_h5 | combo 5
y_h5 | combo 6
y_h5 | combo 7
y_h5 | combo 8
y_h5 | combo 9
y_h5 | combo 10
y_h5 | combo

y_h1 prefers a linear SVR;   
y_h3, y_h5, y_h7 prefer RBF SVR;   
so our SVR behavior is not the same across horizons, which is completely reasonable and supports our choice to build four separate tuned models  
so I proceeded with further tuning for each model

In [8]:
# define helper function to tune one target column with given param grid and train/val splits
def refine_one_target(target_col, param_grid, X_train, X_val, train_sub, val_sub):
    y_train = train_sub[target_col]
    y_val = val_sub[target_col]

    all_params = list(ParameterGrid(param_grid))
    results = []

    best_rmse = np.inf
    best_result = None

    print(f"\n===== Refining {target_col} =====")
    print(f"Total combos: {len(all_params)}")

    for i, params in enumerate(all_params, start=1):
        print(f"{target_col} | combo {i}/{len(all_params)} | {params}")

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR())
        ])

        model.set_params(**params)
        model.fit(X_train, y_train)

        val_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        mae = mean_absolute_error(y_val, val_pred)

        row = {
            "target": target_col,
            "rmse_val": rmse,
            "mae_val": mae,
            **params
        }
        results.append(row)

        if rmse < best_rmse:
            best_rmse = rmse
            best_result = row

    results_df = pd.DataFrame(results).sort_values("rmse_val").reset_index(drop=True)

    print(f"\nBest result for {target_col}:")
    print("RMSE:", round(best_result["rmse_val"], 6))
    print("MAE :", round(best_result["mae_val"], 6))
    print("Params:", {k: v for k, v in best_result.items() if k.startswith("svr__")})

    return results_df, best_result

In [9]:
# stage 2 tuning: expand C and epsilon around best values from stage 1, and add gamma values for rbf kernel
# stage 2 for y_h1
PARAM_GRID_YH1 = {
    "svr__kernel": ["linear"],
    "svr__C": [0.01, 0.05, 0.1, 0.5, 1],
    "svr__epsilon": [0.05, 0.1, 0.2, 0.3]
}

results_yh1_refined, best_yh1_refined = refine_one_target(
    target_col="y_h1",
    param_grid=PARAM_GRID_YH1,
    X_train=X_train,
    X_val=X_val,
    train_sub=train_sub,
    val_sub=val_sub
)

results_yh1_refined.head(10)


===== Refining y_h1 =====
Total combos: 20
y_h1 | combo 1/20 | {'svr__C': 0.01, 'svr__epsilon': 0.05, 'svr__kernel': 'linear'}
y_h1 | combo 2/20 | {'svr__C': 0.01, 'svr__epsilon': 0.1, 'svr__kernel': 'linear'}
y_h1 | combo 3/20 | {'svr__C': 0.01, 'svr__epsilon': 0.2, 'svr__kernel': 'linear'}
y_h1 | combo 4/20 | {'svr__C': 0.01, 'svr__epsilon': 0.3, 'svr__kernel': 'linear'}
y_h1 | combo 5/20 | {'svr__C': 0.05, 'svr__epsilon': 0.05, 'svr__kernel': 'linear'}
y_h1 | combo 6/20 | {'svr__C': 0.05, 'svr__epsilon': 0.1, 'svr__kernel': 'linear'}
y_h1 | combo 7/20 | {'svr__C': 0.05, 'svr__epsilon': 0.2, 'svr__kernel': 'linear'}
y_h1 | combo 8/20 | {'svr__C': 0.05, 'svr__epsilon': 0.3, 'svr__kernel': 'linear'}
y_h1 | combo 9/20 | {'svr__C': 0.1, 'svr__epsilon': 0.05, 'svr__kernel': 'linear'}
y_h1 | combo 10/20 | {'svr__C': 0.1, 'svr__epsilon': 0.1, 'svr__kernel': 'linear'}
y_h1 | combo 11/20 | {'svr__C': 0.1, 'svr__epsilon': 0.2, 'svr__kernel': 'linear'}
y_h1 | combo 12/20 | {'svr__C': 0.1, 'svr

,target,rmse_val,mae_val,svr__C,svr__epsilon,svr__kernel
0,y_h1,0.776007,0.590539,0.01,0.20,linear
1,y_h1,0.782702,0.603872,0.05,0.10,linear
2,y_h1,0.782941,0.594982,0.01,0.30,linear
3,y_h1,0.786298,0.607692,0.05,0.05,linear
4,y_h1,0.789574,0.610290,1.00,0.20,linear
5,y_h1,0.789690,0.611787,0.10,0.10,linear
6,y_h1,0.790199,0.611246,0.10,0.05,linear
7,y_h1,0.792207,0.614113,0.50,0.10,linear
8,y_h1,0.792906,0.606035,0.10,0.20,linear
9,y_h1,0.792980,0.604627,0.05,0.20,linear


In [10]:
# stage 2 for y_h3
PARAM_GRID_YH3 = {
    "svr__kernel": ["rbf"],
    "svr__C": [0.5, 1, 2, 5],
    "svr__epsilon": [0.05, 0.1, 0.2],
    "svr__gamma": ["scale", 0.01, 0.05]
}

results_yh3_refined, best_yh3_refined = refine_one_target(
    target_col="y_h3",
    param_grid=PARAM_GRID_YH3,
    X_train=X_train,
    X_val=X_val,
    train_sub=train_sub,
    val_sub=val_sub
)

results_yh3_refined.head(10)


===== Refining y_h3 =====
Total combos: 36
y_h3 | combo 1/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h3 | combo 2/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h3 | combo 3/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h3 | combo 4/36 | {'svr__C': 0.5, 'svr__epsilon': 0.1, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h3 | combo 5/36 | {'svr__C': 0.5, 'svr__epsilon': 0.1, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h3 | combo 6/36 | {'svr__C': 0.5, 'svr__epsilon': 0.1, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h3 | combo 7/36 | {'svr__C': 0.5, 'svr__epsilon': 0.2, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h3 | combo 8/36 | {'svr__C': 0.5, 'svr__epsilon': 0.2, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h3 | combo 9/36 | {'svr__C': 0.5, 'svr__epsilon': 0.2, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h3 | combo 10/36 | {'svr__C': 1, 'svr__epsilon': 0.

,target,rmse_val,mae_val,svr__C,svr__epsilon,svr__gamma,svr__kernel
0,y_h3,0.875690,0.664273,0.5,0.10,scale,rbf
1,y_h3,0.876415,0.666147,0.5,0.05,scale,rbf
2,y_h3,0.877084,0.663323,0.5,0.20,scale,rbf
3,y_h3,0.877512,0.665275,1.0,0.10,scale,rbf
4,y_h3,0.877884,0.666556,1.0,0.05,scale,rbf
5,y_h3,0.880674,0.665845,1.0,0.20,scale,rbf
6,y_h3,0.888818,0.676361,2.0,0.05,scale,rbf
7,y_h3,0.889758,0.677128,2.0,0.10,scale,rbf
8,y_h3,0.892850,0.679752,2.0,0.20,scale,rbf
9,y_h3,0.898581,0.684177,1.0,0.20,0.01,rbf


In [11]:
# stage 2 for y_h5
PARAM_GRID_YH5 = {
    "svr__kernel": ["rbf"],
    "svr__C": [0.5, 1, 2, 5],
    "svr__epsilon": [0.005, 0.01, 0.05],
    "svr__gamma": ["scale", 0.01, 0.05]
}

results_yh5_refined, best_yh5_refined = refine_one_target(
    target_col="y_h5",
    param_grid=PARAM_GRID_YH5,
    X_train=X_train,
    X_val=X_val,
    train_sub=train_sub,
    val_sub=val_sub
)

results_yh5_refined.head(10)


===== Refining y_h5 =====
Total combos: 36
y_h5 | combo 1/36 | {'svr__C': 0.5, 'svr__epsilon': 0.005, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h5 | combo 2/36 | {'svr__C': 0.5, 'svr__epsilon': 0.005, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h5 | combo 3/36 | {'svr__C': 0.5, 'svr__epsilon': 0.005, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h5 | combo 4/36 | {'svr__C': 0.5, 'svr__epsilon': 0.01, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h5 | combo 5/36 | {'svr__C': 0.5, 'svr__epsilon': 0.01, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h5 | combo 6/36 | {'svr__C': 0.5, 'svr__epsilon': 0.01, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h5 | combo 7/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h5 | combo 8/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h5 | combo 9/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h5 | combo 10/36 | {'svr__C': 1, 'svr__eps

,target,rmse_val,mae_val,svr__C,svr__epsilon,svr__gamma,svr__kernel
0,y_h5,0.810254,0.618914,0.5,0.050,scale,rbf
1,y_h5,0.810837,0.622426,1.0,0.005,scale,rbf
2,y_h5,0.810842,0.622410,1.0,0.010,scale,rbf
3,y_h5,0.811249,0.622568,1.0,0.050,scale,rbf
4,y_h5,0.812035,0.620454,0.5,0.010,scale,rbf
5,y_h5,0.812422,0.620745,0.5,0.005,scale,rbf
6,y_h5,0.826443,0.635212,2.0,0.050,scale,rbf
7,y_h5,0.827395,0.636124,2.0,0.010,scale,rbf
8,y_h5,0.827480,0.636197,2.0,0.005,scale,rbf
9,y_h5,0.833291,0.638662,1.0,0.005,0.01,rbf


In [12]:
# stage 2 for y_h7
PARAM_GRID_YH7 = {
    "svr__kernel": ["rbf"],
    "svr__C": [0.5, 1, 2, 5],
    "svr__epsilon": [0.005, 0.01, 0.05],
    "svr__gamma": ["scale", 0.01, 0.05]
}

results_yh7_refined, best_yh7_refined = refine_one_target(
    target_col="y_h7",
    param_grid=PARAM_GRID_YH7,
    X_train=X_train,
    X_val=X_val,
    train_sub=train_sub,
    val_sub=val_sub
)

results_yh7_refined.head(10)


===== Refining y_h7 =====
Total combos: 36
y_h7 | combo 1/36 | {'svr__C': 0.5, 'svr__epsilon': 0.005, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h7 | combo 2/36 | {'svr__C': 0.5, 'svr__epsilon': 0.005, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h7 | combo 3/36 | {'svr__C': 0.5, 'svr__epsilon': 0.005, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h7 | combo 4/36 | {'svr__C': 0.5, 'svr__epsilon': 0.01, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h7 | combo 5/36 | {'svr__C': 0.5, 'svr__epsilon': 0.01, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h7 | combo 6/36 | {'svr__C': 0.5, 'svr__epsilon': 0.01, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h7 | combo 7/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 'scale', 'svr__kernel': 'rbf'}
y_h7 | combo 8/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 0.01, 'svr__kernel': 'rbf'}
y_h7 | combo 9/36 | {'svr__C': 0.5, 'svr__epsilon': 0.05, 'svr__gamma': 0.05, 'svr__kernel': 'rbf'}
y_h7 | combo 10/36 | {'svr__C': 1, 'svr__eps

,target,rmse_val,mae_val,svr__C,svr__epsilon,svr__gamma,svr__kernel
0,y_h7,0.839859,0.633501,1.0,0.005,scale,rbf
1,y_h7,0.839981,0.633722,1.0,0.010,scale,rbf
2,y_h7,0.840240,0.634770,1.0,0.050,scale,rbf
3,y_h7,0.840454,0.629800,0.5,0.050,scale,rbf
4,y_h7,0.840645,0.629960,0.5,0.010,scale,rbf
5,y_h7,0.840750,0.630025,0.5,0.005,scale,rbf
6,y_h7,0.854477,0.643629,2.0,0.050,scale,rbf
7,y_h7,0.856637,0.645136,2.0,0.010,scale,rbf
8,y_h7,0.856924,0.645306,2.0,0.005,scale,rbf
9,y_h7,0.863791,0.653584,2.0,0.050,0.01,rbf


In [13]:
best_params_refined = {
    "y_h1": {k: v for k, v in best_yh1_refined.items() if k.startswith("svr__")},
    "y_h3": {k: v for k, v in best_yh3_refined.items() if k.startswith("svr__")},
    "y_h5": {k: v for k, v in best_yh5_refined.items() if k.startswith("svr__")},
    "y_h7": {k: v for k, v in best_yh7_refined.items() if k.startswith("svr__")}
}

best_params_refined

{'y_h1': {'svr__C': 0.01, 'svr__epsilon': 0.2, 'svr__kernel': 'linear'},
 'y_h3': {'svr__C': 0.5,
  'svr__epsilon': 0.1,
  'svr__gamma': 'scale',
  'svr__kernel': 'rbf'},
 'y_h5': {'svr__C': 0.5,
  'svr__epsilon': 0.05,
  'svr__gamma': 'scale',
  'svr__kernel': 'rbf'},
 'y_h7': {'svr__C': 1,
  'svr__epsilon': 0.005,
  'svr__gamma': 'scale',
  'svr__kernel': 'rbf'}}

gamma="scale" keeps winning for all the RBF models;  
the best C values are all small, which suggests the model prefers a fairly regularized fit

y_h1: 0.789690 → 0.776007  
improvement = 0.013683 (~1.73%)  
meaningful improviment  

y_h3: 0.877512 → 0.875690  
improvement = 0.001822 (~0.21%)  
y_h5: 0.810842 → 0.810254  
improvement = 0.000588 (~0.07%)  
y_h7: 0.839981 → 0.839859  
improvement = 0.000122 (~0.01%)  
tiny improvement  

In [ ]:
#final refiningment for h1 around best values from stage 2
PARAM_GRID_YH1_FINAL = {
    "svr__kernel": ["linear"],
    "svr__C": [0.009, 0.01, 0.011,0.012],
    "svr__epsilon": [0.18, 0.19, 0.2, 0.21, 0.22]
}

In [27]:
results_yh1_refined_final, best_yh1_refined_final = refine_one_target(
    target_col="y_h1",
    param_grid=PARAM_GRID_YH1_FINAL,
    X_train=X_train,
    X_val=X_val,
    train_sub=train_sub,
    val_sub=val_sub
)

results_yh1_refined_final.head(10)


===== Refining y_h1 =====
Total combos: 20
y_h1 | combo 1/20 | {'svr__C': 0.009, 'svr__epsilon': 0.18, 'svr__kernel': 'linear'}
y_h1 | combo 2/20 | {'svr__C': 0.009, 'svr__epsilon': 0.19, 'svr__kernel': 'linear'}
y_h1 | combo 3/20 | {'svr__C': 0.009, 'svr__epsilon': 0.2, 'svr__kernel': 'linear'}
y_h1 | combo 4/20 | {'svr__C': 0.009, 'svr__epsilon': 0.21, 'svr__kernel': 'linear'}
y_h1 | combo 5/20 | {'svr__C': 0.009, 'svr__epsilon': 0.22, 'svr__kernel': 'linear'}
y_h1 | combo 6/20 | {'svr__C': 0.01, 'svr__epsilon': 0.18, 'svr__kernel': 'linear'}
y_h1 | combo 7/20 | {'svr__C': 0.01, 'svr__epsilon': 0.19, 'svr__kernel': 'linear'}
y_h1 | combo 8/20 | {'svr__C': 0.01, 'svr__epsilon': 0.2, 'svr__kernel': 'linear'}
y_h1 | combo 9/20 | {'svr__C': 0.01, 'svr__epsilon': 0.21, 'svr__kernel': 'linear'}
y_h1 | combo 10/20 | {'svr__C': 0.01, 'svr__epsilon': 0.22, 'svr__kernel': 'linear'}
y_h1 | combo 11/20 | {'svr__C': 0.011, 'svr__epsilon': 0.18, 'svr__kernel': 'linear'}
y_h1 | combo 12/20 | {'svr

,target,rmse_val,mae_val,svr__C,svr__epsilon,svr__kernel
0,y_h1,0.773715,0.588574,0.011,0.19,linear
1,y_h1,0.774271,0.588960,0.012,0.19,linear
2,y_h1,0.774724,0.589265,0.010,0.19,linear
3,y_h1,0.774937,0.588600,0.011,0.20,linear
4,y_h1,0.775058,0.588567,0.011,0.21,linear
5,y_h1,0.775219,0.590179,0.009,0.19,linear
6,y_h1,0.775276,0.588656,0.012,0.20,linear
7,y_h1,0.775490,0.589552,0.010,0.21,linear
8,y_h1,0.775787,0.590286,0.011,0.18,linear
9,y_h1,0.776007,0.590539,0.010,0.20,linear


A final local refinement for the 1-day horizon produced a small additional gain, improvement: 0.2292%  
improving validation RMSE from 0.7760 to 0.7737. stop tuning here.

In [ ]:
# overwrite y_h1 refined objects with final micro-refined version
results_yh1_refined = results_yh1_refined_final.copy()
best_yh1_refined = best_yh1_refined_final.copy()

all_refined_results = pd.concat([
    results_yh1_refined, #update here for yh1 only
    results_yh3_refined,
    results_yh5_refined,
    results_yh7_refined
], ignore_index=True)

# final locked params
final_svr_params = {
    "y_h1": {
        "svr__C": 0.011,
        "svr__epsilon": 0.19,
        "svr__kernel": "linear"
    },
    "y_h3": {
        "svr__C": 0.5,
        "svr__epsilon": 0.1,
        "svr__gamma": "scale",
        "svr__kernel": "rbf"
    },
    "y_h5": {
        "svr__C": 0.5,
        "svr__epsilon": 0.05,
        "svr__gamma": "scale",
        "svr__kernel": "rbf"
    },
    "y_h7": {
        "svr__C": 1,
        "svr__epsilon": 0.005,
        "svr__gamma": "scale",
        "svr__kernel": "rbf"
    }
}
# summary table of final chosen model per horizon
final_svr_summary = pd.DataFrame([
    {
        "target": "y_h1",
        "rmse_val": best_yh1_refined["rmse_val"],
        "mae_val": best_yh1_refined["mae_val"],
        **final_svr_params["y_h1"]
    },
    {
        "target": "y_h3",
        "rmse_val": best_yh3_refined["rmse_val"],
        "mae_val": best_yh3_refined["mae_val"],
        **final_svr_params["y_h3"]
    },
    {
        "target": "y_h5",
        "rmse_val": best_yh5_refined["rmse_val"],
        "mae_val": best_yh5_refined["mae_val"],
        **final_svr_params["y_h5"]
    },
    {
        "target": "y_h7",
        "rmse_val": best_yh7_refined["rmse_val"],
        "mae_val": best_yh7_refined["mae_val"],
        **final_svr_params["y_h7"]
    }
])

# all_refined_results.to_csv(os.path.join(OUT_DIR, "svr_refined_tuning_results.csv"), index=False)
# this is just results of all combo, not useful for us 
final_svr_summary.to_csv(os.path.join(OUT_DIR, "svr_final_summary.csv"), index=False)
print("Saved refined best params.")


Saved refined best params.
